# Random Forest and XGBoost Models Script

## This script accomplishes the following:

In this script, we will run 2 models, to predict the number of conflicts in each ZIP code, in each month.
The predictions are generated in a traditional ML way. We split the data into train and test sets, train the random forest on the former and  make the predicictions on the later.
Hence, we mask data that we "already saw", and then evaluate the models performance by taking the deviation from our predictions to the actual data.

In this script, we **predict**, we don't **forecast**, as the data is present and has already taken place, hence giving us a way to actually evaluate the model.

## Find the following steps below:

1. **Random Forest Model**

- Define features and label (number of conflicts, for each ZIP in PA, for each month)
- Perform 5-fold time-series-cross-validation
- Split the data 70/30 train and test
- Normalize the sets
- Define the rf regressor and fit
- Perform feature importance analysis
- Run a Grid Search to find the optimal parameter values
- Re-fit the model
- Make predictions
- Evaluation of Predictions

2. **XGBoost Model**

- Define the xgb regressor and fit
- Perform feature importance analysis
- Run a Grid Search to find the optimal parameter values
- Re-fit the model
- Make predictions
- Evaluation of Predictions

In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

In [35]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

### Load the data

In [36]:
# Load the data

data = pd.read_csv(os.path.join(project_root_dir, 'Dataset/acs_conflict_combined_cleaned.csv'))

#data = pd.read_csv(project_root_dir, 'acs_conflict_combined_cleaned.csv')

## 1. Random Forest

### Features and labels, cross-validation, split and normalization

In [37]:
# Define features and label

X = data.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y = data['event_count']

In [38]:
# Time Series Cross Validation, 5 folds.

tscv = TimeSeriesSplit(n_splits=5)

In [39]:
# Split the dataset into training and test sets
# 70/30 split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

In [40]:
# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Define the regressor and run

In [41]:
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=100)

In [45]:
rf_regressor.fit(X_train_scaled, Y_train)

RandomForestRegressor(random_state=100)

### Feature importance

In [ ]:
# Which are the most important features?

feature_importances = rf_regressor.feature_importances_
sorted_indices = np.argsort(feature_importances)[::-1]

In [ ]:
'''
# Create the plot
plt.figure(figsize=(10, 8))
plt.title('Feature Importances', fontsize=20)
plt.bar(range(len(feature_importances)), feature_importances[sorted_indices], align='center', alpha=0.5)
plt.xticks(range(len(feature_importances)), feature_names[sorted_indices], rotation=90)
plt.ylabel('Relative Importance', fontsize=15)
plt.xlabel('Feature', fontsize=15)
plt.tight_layout()  # Adjusts the plot to ensure everything fits without overlapping
plt.show()
'''

### Run Grid Search to find the best parameters

In [24]:
# Grid Search for best parameters

# Adjust number of trees in forest, max depth of trees, and minimum samples per leaf

# n_estimators: Number of trees in the forest, to take into account for the prediction
# max_depth: Maximum depth of the tree. From root to leaf, basically controls complexity of the tree and can lead/prevent overfitting. The more splits, the more complex the tree and the more likely to fit noise.
# min_samples_split: Minimum number of samples required to split an internal node


# Prevents overfitting and improves the generalization of the model

param_grid = {
    'n_estimators': [10, 20, 40],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 7]
}

In [25]:
cv_rf = GridSearchCV(estimator=rf_regressor, param_grid=param_grid, cv=5)
cv_rf.fit(X_train_scaled, Y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=100),
             param_grid={'max_depth': [None, 10, 20, 30],
                         'min_samples_split': [2, 5, 7],
                         'n_estimators': [10, 20, 40]})

## Run the predictions

### Without Best Parameters

In [46]:
Y_pred = rf_regressor.predict(X_test_scaled)

predictions = pd.DataFrame({'Actual': Y_test, 'Predicted': Y_pred})
print(Y_pred)

[0.00622858 0.00670006 0.00716701 ... 0.4348492  0.00636438 0.00622858]


In [47]:
mse = mean_squared_error(Y_test, Y_pred)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.07406755829726698


### Including the Best Parameters

In [50]:
Y_pred_gs = cv_rf.predict(X_test_scaled)

predictions_gs = pd.DataFrame({'Actual': Y_test, 'Predicted': Y_pred_gs})
print(Y_pred_gs)

[0.00574218 0.00571976 0.00613604 ... 0.27918392 0.00569096 0.00574218]


In [49]:
mse_gs = mean_squared_error(Y_test, Y_pred_gs)

print('Mean Squared Error:', mse_gs)

Mean Squared Error: 0.0781439381243572


### Show predictions and model evaluations

In [28]:
ZIP_results = pd.merge(predictions, data[['ZCTA', 'Month']], left_index=True, right_index=True)


In [29]:
print(ZIP_results)

       Actual  Predicted     ZCTA  Month
2232      0.0   0.005742  15364.0      1
13760     0.0   0.005720  17702.0      9
559       0.0   0.006136  15061.0      8
7310      0.0   0.005742  16238.0      3
352       0.0   0.005720  15042.0      5
...       ...        ...      ...    ...
13909     0.0   0.005742  17742.0      2
10454     0.0   0.005742  16922.0      3
20010     0.0   0.279184  19104.0      7
20584     0.0   0.005691  19301.0      5
14396     0.0   0.005742  17832.0      9

[6599 rows x 4 columns]


In [30]:
ZIP_results = ZIP_results[ZIP_results['Predicted'] >= 1]

In [31]:
mse = mean_squared_error(Y_test, Y_pred_gs)

print('Mean Squared Error:', mse)

Mean Squared Error: 0.0781439381243572


In [ ]:
# results_df = pd.DataFrame(columns=['Model', 'RMSLE', 'RMSE', 'MSE', 'MAE'])


## 2. XGboost

### Fit the model

In [56]:
xgb = XGBRegressor(use_label_encoder = False, eval_metric = 'logloss', random_state = 1 )
xgb.fit(X_train_scaled, Y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='logloss',
             feature_types=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, random_state=1, ...)

### Run Grid Search to find the best parameters:

In [58]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9]
}

In [59]:
# Initialize XGB and GridSearch
grid_search = GridSearchCV(xgb, param_grid, scoring='accuracy', cv=5, verbose=1)
grid_search.fit(X_train_scaled, Y_train)

# Print the best parameters
best_params = grid_search.best_params_
print("Best parameters:", best_params)

Fitting 5 folds for each of 5120 candidates, totalling 25600 fits


c:\Users\danie\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py:988: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\danie\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_validation.py", line 977, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_scorer.py", line 253, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\danie\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_scorer.py", line 350, in _score
    return self._sign * self.

KeyboardInterrupt: 

### Re-run the model with optimal parameter values

In [ ]:
# Re-train the model with the best parameters found above
xgb_optimized = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state = 1)
xgb_optimized.fit(X_train_scaled, Y_train)

# Make predictions
y_pred_optimized = xgb_optimized.predict(X_test_scaled)

# Accurcary score
accuracy_optimized = accuracy_score(Y_test, y_pred_optimized)
print("Test set accuracy with optimized parameters: {:.2f}%".format(accuracy_optimized * 100))

### Run feature importance analysis

In [ ]:
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure()
plt.title('Feature Importances')
plt.bar(range(X_train.shape[1]), importances[indices], color='r', align='center')
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.show()

### Find best number of features

In [ ]:
# Empty list to store and run different cv scores
cv_scores = []

for i in range(1, len(indices) + 1):
    # Select the top 'i' features
    top_features = [X_train.columns[indices[j]] for j in range(i)]
    
    # Create the training data using the selected features
    X_train_top = X_train_scaled[:, indices[:i]]
    
    # Initialize the model with the best parameters
    model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
    
    # Perform cv at 5 and store the mean score
    scores = cross_val_score(model, X_train_top, Y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Which number of features gives the highest cv score?
optimal_features = np.argmax(cv_scores) + 1 
optimal_score = cv_scores[optimal_features - 1]

print(f"Optimal number of features: {optimal_features}")
print(f"Highest CV accuracy: {optimal_score:.2f}%")

### Plot

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(indices) + 1), cv_scores, marker='o')
plt.xlabel('Number of Features')
plt.ylabel('CV Accuracy')
plt.title('CV Performance vs. Number of Features')
plt.show()

### Re-run with best features

In [ ]:


# Select opimal features
top_features_indices = indices[:optimal_features]
X_train_optimal = X_train_scaled[:, top_features_indices]
X_test_optimal = X_test_scaled[:, top_features_indices]

# Train the model using the selected subset of features
model_optimal = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
model_optimal.fit(X_train_optimal, y_train)

# Evaluate the model on the test set
y_pred_optimal = model_optimal.predict(X_test_optimal)
accuracy_optimal = accuracy_score(y_test, y_pred_optimal)

print(f"Test set accuracy with optimal number of features ({optimal_features}): {accuracy_optimal:.2f}%")